<img src='../OUTILS/bandeau_MF.png' align='right' width='100%'/>

# <div style='background-color: #27ae60; color: white; padding: 20px; border-radius: 10px; text-align: center;'>🌍🛰️Manipulation de données satellitaires - Format NetCDF</div>

### 🎯Analyser un fichier NETCDF
### 🎯Produire une image
### 🎯Appliquer une palette
### 🎯Reprojeter un planisphère

## Workflow : lignes de commandes bash : GDAL & ImageMagick 

Ce TP utilise en grande partie les logiciels de la librairie **"gdal"** qui signifie : *geospatial data abstraction library*. Cette librairie est extrêmement utile pour manipuler les données issues des satellites météorologiques.

Pour ce TP nous allons utiliser des fichiers **NetCDF** de température de surface de la mer issus de la production Copernicus.  </br>


<div class="alert alert-info" role="alert">
<h3> ⚙️ Initialisation de l'environnement</h3>
Tout d'abord, il faut procéder à l'importation des librairies nécessaires à ce TP.
</div>

In [ ]:
from datetime import datetime
import sys
import os
from osgeo import gdal
from PIL import Image
import subprocess
from IPython.display import display,HTML
os.environ['PATH'] = f"/opt/conda/env_MF_teledetection/bin:{os.environ['PATH']}" 
os.environ['PATH'] = f"~/.conda/envs/env_MF_teledetection/bin:{os.environ['PATH']}"
os.environ['GDAL_DATA'] = '/opt/conda/env_MF_teledetection/share/gdal'
os.environ['PROJ_LIB'] = '/opt/conda/env_MF_teledetection/share/proj'

<div class="alert alert-info alert-success">
<h3> 1 - 🔎 Produire une image TIF à partir d'un NetCDF de SST    </h3>
</div>

In [ ]:
INPUT="/stockage/DATA/NetCDF/OSTIA/202604.nc"
print(f"Fichier source configuré : {INPUT}")

In [ ]:
cd ~/MF_DATA_MANIPULATION

In [ ]:
!pwd

In [ ]:
!mkdir -p RESULTS
output = 'RESULTS'

In [ ]:
!gdalinfo $INPUT

# 🌊 **Analyse détaillée du fichier NetCDF OSTIA**
### *Sea Surface Temperature (SST) - UK Met Office*

---

## 📁 **Informations générales**

| Propriété | Valeur |
|-----------|--------|
| 📛 **Nom** | `202604.nc` |
| 🏛️ **Institution** | UKMO (UK Met Office) |
| 🛰️ **Produit** | OSTIA (Operational SST and Ice Analysis) |
| 📐 **Format** | NetCDF-4, CF-1.11 |
| 🎯 **Usage** | Température de surface de la mer (fondation) |

---

## 🌍 **Géoréférencement**

| Paramètre | Valeur |
|-----------|--------|
| 🗺️ **Projection** | Géographique WGS84 (directe) |
| 📏 **Résolution** | **0.05°** (≈ 5.5 km à l'équateur) |
| 📊 **Dimensions** | 7200 (lon) × 3600 (lat) |
| 🔄 **Emprise** | Globe entier : -180° ↔ 180° lon, -90° ↔ 90° lat |
| 📍 **Ordre coord.** | **Longitude puis latitude** (X,Y) |

> 💡 *Pas de projection complexe : les coordonnées sont directement en degrés !*

---

## 🌡️ **Variable principale : `analysed_sst`**

| Propriété | Valeur |
|-----------|--------|
| 🏷️ **Nom complet** | Analysed sea surface temperature |
| 📊 **Type** | Float32 (nombres décimaux 32 bits) |
| 📐 **Unité** | **Kelvin** (K) |
| ✅ **Min valide** | 270.15 K (-3.0°C) |
| ❌ **Max valide** | 318.15 K (45.0°C) |
| 🚫 **NoData** | `nan` (Not a Number) |

### 🔁 Conversion en Celsius 
```python
# Formules de conversion
Celsius = Kelvin - 273.15

Production du TIF, et visualisation

In [ ]:
!gdal_translate -scale 270 318 0 255 -ot Byte  $INPUT {output}/202604_sst_ostia.tif 2>/dev/null
!convert -resize 1000x500 {output}/202604_sst_ostia.tif {output}/202604_sst_ostia.jpg 2>/dev/null 
im1 = Image.open(output + '/202604_sst_ostia.jpg', 'r')
display(im1)

Il est possible de modifier la dynamique si nécessaire, par exemple pour resserer la plage de -2,15 à 32,85 °C (271 à 306 K)

In [ ]:
!gdal_translate -scale 271 306 0 255 -ot Byte  $INPUT {output}/202604_sst_ostia_271_306.tif 2>/dev/null
!convert -resize 1000x500 {output}/202604_sst_ostia_271_306.tif {output}/202604_sst_ostia_271_306.jpg 2>/dev/null 
im2 = Image.open(output + '/202604_sst_ostia.jpg', 'r')
display(im2)

Appliquer une palette à un TIF en nuance de gris : commande gdaldem color-relief 

Palette : Voir le détail

In [ ]:
!gdal_translate $INPUT {output}/202604_sst_ostia_271_306.tif 2>/dev/null
!gdaldem color-relief {output}/202604_sst_ostia_271_306.tif OUTILS/palette_SST.txt {output}/202604_sst_ostia_palette.tif
!convert -resize 1000x500 {output}/202604_sst_ostia_palette.tif {output}/202604_sst_ostia_palette.jpg
im3 = Image.open(output + '/202604_sst_ostia_palette.jpg', 'r')
display(im3)

Ajoutons les terres
Nous allons utiliser les Bluemarble de la NASA
Décrire la Bluemarble ici 

Visualisons la bluemarble

In [ ]:
!convert -resize 1000x500  /stockage/DATA/bluemarble/bluemarble_transp_04.png {output}/bluemarble_transp_04_1000x500.png
im4 = Image.open(output + '/bluemarble_transp_04_1000x500.png', 'r')
display(im4)

Fusion des 2 images

In [ ]:
!composite -quiet -geometry +0+0 {output}/bluemarble_transp_04_1000x500.png {output}/202604_sst_ostia_palette.jpg {output}/202604_sst_ostia_bluemarble.jpg
im5 = Image.open(output + '/202604_sst_ostia_bluemarble.jpg', 'r')
display(im5)

Nous allons reprojeter cette image. </br>
Cependant avec la fusion Magick et la creation d'un jpg, nous avons perdu le georeferencement.</br>
Il faut donc géoréférencer l'image et garder le format TIF
Nous allons refaire le planisphère avec une meilleur résolution:

In [ ]:
dimension_planisphere = "3000x1500"

In [ ]:
!gdal_translate $INPUT {output}/202604_sst_ostia_271_306.tif 2>/dev/null
!gdaldem color-relief {output}/202604_sst_ostia_271_306.tif OUTILS/palette_SST.txt {output}/202604_sst_ostia_palette.tif
!convert -resize {dimension_planisphere} {output}/202604_sst_ostia_palette.tif {output}/202604_sst_ostia_palette_{dimension_planisphere}.jpg
!convert -resize {dimension_planisphere}  /stockage/DATA/bluemarble/bluemarble_transp_04.png {output}/bluemarble_transp_04_{dimension_planisphere}.png
!composite -quiet -geometry +0+0 {output}/bluemarble_transp_04_{dimension_planisphere}.png  {output}/202604_sst_ostia_palette_{dimension_planisphere}.jpg {output}/202604_sst_ostia_bluemarble_{dimension_planisphere}.tif
!gdal_translate -of GTiff -a_srs EPSG:4326 -a_ullr -180 90 180 -90 {output}/202604_sst_ostia_bluemarble_{dimension_planisphere}.tif {output}/202604_sst_ostia_bluemarble.tif

Nous allons faire quelques projection différentes


| Projection | Type | Caractéristique principale | Inconvénient majeur | Commande GDAL |
|------------|------|---------------------------|---------------------|---------------|
| **Peters** | Cylindrique équivalente | Conserve les surfaces (les pays du Sud ne sont pas minimisés) | Distorsion verticale extrême aux pôles et à l'équateur | `gdalwarp -t_srs "+proj=cea +lon_0=0 +lat_ts=0 +datum=WGS84 +units=m +no_defs" input.tif peters_output.tif` |
| **Robinson** | Pseudocylindrique (compromis) | Compromis harmonieux entre forme, surface et distorsion des angles | N'est ni​ conforme, ni équivalente | `gdalwarp -t_srs "+proj=robin +lon_0=0 +datum=WGS84 +units=m +no_defs" input.tif robinson_output.tif` |
| **Mollweide** | Pseudocylindrique équivalente | Conserve les surfaces, forme elliptique élégante | Forte distorsion des angles près des bords | `gdalwarp -t_srs "+proj=moll +lon_0=0 +datum=WGS84 +units=m +no_defs" input.tif mollweide_output.tif` |

**Notes :**
- Peters : Cylindrical Equal Area (cea) avec `lat_ts=0`
- Robinson : utilisée par National Geographic (1988-1998)
- Mollweide : idéale pour cartes thématiques globales
- Aucune des trois n'a de code EPSG officiel 


Avec la proj Peters

In [ ]:
!gdalwarp -overwrite -t_srs "+proj=cea +lon_0=0 +lat_ts=45 +x_0=0 +y_0=0 +ellps=WGS84 +units=m +no_defs" -r cubic {output}/202604_sst_ostia_bluemarble.tif {output}/202604_sst_ostia_bluemarble_peters.tif -co COMPRESS=LZW >/dev/null 2>&1
!convert -resize 1000 {output}/202604_sst_ostia_bluemarble_peters.tif  {output}/202604_sst_ostia_bluemarble_peters.jpg  >/dev/null 2>&1
im6 = Image.open(output + '/202604_sst_ostia_bluemarble_peters.jpg', 'r')
display(im6)

Avec la proj Robinson

In [ ]:
!gdalwarp -t_srs "+proj=robin +lon_0=0 +x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs" -r cubic  {output}/202604_sst_ostia_bluemarble.tif {output}/202604_sst_ostia_bluemarble_robinson.tif -co COMPRESS=LZW  >/dev/null 2>&1
!convert -resize 1000 {output}/202604_sst_ostia_bluemarble_robinson.tif  {output}/202604_sst_ostia_bluemarble_robinson.jpg  >/dev/null 2>&1
im7 = Image.open(output + '/202604_sst_ostia_bluemarble_robinson.jpg', 'r')
display(im7)

Avec la proj Mollweide

In [ ]:
!gdalwarp -t_srs "+proj=moll +lon_0=0 +datum=WGS84 +units=m +no_defs"  -r cubic  {output}/202604_sst_ostia_bluemarble.tif {output}/202604_sst_ostia_bluemarble_mollweide.tif -co COMPRESS=LZW  >/dev/null 2>&1
!convert -resize 1000 {output}/202604_sst_ostia_bluemarble_mollweide.tif  {output}/202604_sst_ostia_bluemarble_mollweide.jpg  >/dev/null 2>&1
im8 = Image.open(output + '/202604_sst_ostia_bluemarble_mollweide.jpg', 'r')
display(im8)

REprojeter avec une vue geostationnaire (valeurs MTGI1 ici)

La position en longitude du satellite est indiquée ave l'option +lon_0

In [ ]:
!gdalwarp -overwrite -r cubic -t_srs '+proj=geos +a=6378169 +b=6356583.8 +lon_0=0 +h=35785834 +x_0=0 +y_0=0' -te -5500000 -5500000 5500000 5500000 {output}/202604_sst_ostia_bluemarble.tif {output}/202604_sst_ostia_bluemarble_vue_geos_0.tif
!convert -resize 1000 {output}/202604_sst_ostia_bluemarble_vue_geos_0.tif {output}/202604_sst_ostia_bluemarble_vue_geos_0.jpg  >/dev/null 2>&1
im9 = Image.open(output + '/202604_sst_ostia_bluemarble_vue_geos_0.jpg', 'r')
display(im9)

In [ ]:
!gdalwarp -overwrite -r cubic -t_srs '+proj=ob_tran +o_proj=geos +o_lon_p=0 +o_lat_p=40 +lon_0=-30 +a=6378169 +b=6356583.8 +h=35785834 +x_0=0 +y_0=0' -te -5500000 -5500000 5500000 5500000 {output}/202604_sst_ostia_bluemarble.tif {output}/202604_sst_ostia_bluemarble_vue_geos_0.tif 
!convert -resize 1000 {output}/202604_sst_ostia_bluemarble_vue_geos_0.tif {output}/202604_sst_ostia_bluemarble_vue_geos_0.jpg  >/dev/null 2>&1
im9 = Image.open(output + '/202604_sst_ostia_bluemarble_vue_geos_0.jpg', 'r')
display(im9)

# Projection `ob_tran` (Oblique Transformation) avec GDAL

## Description
`ob_tran` permet de transformer n'importe quelle projection en lui appliquant une rotation/décalage (changement de pôle, point de vue satellite incliné). Pour un satellite géostationnaire, elle simule une latitude non nulle (ex: 40°N) alors que `geos` est limité à l'équateur.

## Syntaxe 


## Options détaillées

| Option | Type | Plage | Description |
|--------|------|-------|-------------|
| `+o_proj` | string | - | Projection d'origine (ex: `geos`, `merc`, `longlat`, `ortho`) |
| `+o_lat_p` | float | -90°..90° | Latitude du nouveau pôle/point d'observation (défaut=0) |
| `+o_lon_p` | float | -180°..180° | Longitude du nouveau pôle (défaut=0) |
| `+lon_0` | float | -180°..180° | Méridien central de la projection résultante (défaut=0) |
| `+o_alpha` | float | 0°..360° | Rotation supplémentaire autour du nouveau pôle (défaut=0) |
| `+k` | float | >0 | Facteur d'échelle (défaut=1) |
| `+x_0`, `+y_0` | float | - | Fausses coordonnées (m) |

## Options héritées de `geos` (si `+o_proj=geos`)

| Option | Valeur standard | Signification |
|--------|-----------------|---------------|
| `+h` | 35785834 m | Altitude satellite |
| `+a` | 6378169 m | Demi-grand axe |
| `+b` | 6356583.8 m | Demi-petit axe |
| `+units` | m | Unités |

## Exemples

Aller plus loin </br> -> 
OSI SAF : https://osi-saf.eumetsat.int/ </br>
Jupyter NoteBook : https://gitlab.eumetsat.int/eumetlab/oceans/ocean-training/sensors/learn-osi-saf-sst

Exemple avec produit Sandwich